In [82]:
import torch

data = torch.load("data/train_blip_features.pt")

features = data["features"]
questions = data["questions"]
labels = data["labels"]

print(features.shape)
print(questions.shape)
print(labels.shape)

torch.Size([1722, 32, 768])
torch.Size([1722, 23])
torch.Size([1722])


In [83]:
from torch.utils.data import Dataset

class DisasterVQADataset(Dataset):

    def __init__(self, pt_file):

        data = torch.load(pt_file)

        self.features = data["features"]
        self.questions = data["questions"]
        self.labels = data["labels"]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):

        return (
            self.features[idx],
            self.questions[idx],
            self.labels[idx]
        )

In [84]:
from torch.utils.data import DataLoader

train_dataset = DisasterVQADataset(
    "data/train_blip_features.pt"
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

features, questions, labels = next(
    iter(train_loader)
)

print(features.shape)
print(questions.shape)
print(labels.shape)

torch.Size([32, 32, 768])
torch.Size([32, 23])
torch.Size([32])


Building the encoder and decoder

In [85]:
import torch.nn as nn

class SemanticEncoder(nn.Module):

    def __init__(self):

        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(768,512),
            nn.ReLU(),

            nn.Linear(512, 256),
            nn.ReLU(),

            nn.Linear(256, 128)
        )


    def forward(self, x):

        return self.encoder(x)

In [86]:
class SemanticDecoder(nn.Module):

    def __init__(self):

        super().__init__()

        self.decoder = nn.Sequential(

            nn.Linear(128, 256),
            nn.ReLU(),

            nn.Linear(256, 512),
            nn.ReLU(),

            nn.Linear(512, 768)
        )

    def forward(self, x):

        return self.decoder(x)

In [87]:
def awgn_channel(tx_signal, snr_db):

    signal_power = torch.mean(
        tx_signal ** 2
    )

    snr_linear = 10 ** (snr_db / 10)

    noise_power = signal_power / snr_linear

    noise_std = torch.sqrt(noise_power)

    noise = torch.randn_like(tx_signal) * noise_std

    return tx_signal + noise

def real_to_complex(x):

    real = x[..., ::2]
    imag = x[..., 1::2]

    return torch.complex(real, imag)

def complex_awgn(complex_signal, snr_db):

    signal_power = torch.mean(
        torch.abs(complex_signal) ** 2
    )

    snr_linear = 10 ** (snr_db / 10)

    noise_power = signal_power / snr_linear

    noise_std = torch.sqrt(noise_power / 2)

    noise_real = (
        torch.randn_like(complex_signal.real)
        * noise_std
    )

    noise_imag = (
        torch.randn_like(complex_signal.imag)
        * noise_std
    )

    noise = torch.complex(
        noise_real,
        noise_imag
    )

    return complex_signal + noise

def complex_to_real(z):

    real = z.real
    imag = z.imag

    return torch.cat(
        [real, imag],
        dim=-1
    )

In [88]:
class QuestionEncoder(nn.Module):

    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=128):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=0
        )

        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True
        )

    def forward(self, questions):

        embedded = self.embedding(
            questions
        )

        outputs, (hidden, cell) = self.lstm(
            embedded
        )

        question_vector = torch.cat(
            (hidden[-2], hidden[-1]),
            dim=1
        )

        return question_vector

In [89]:
class VQAClassifier(nn.Module):

    def __init__(self):

        super().__init__()

        self.classifier = nn.Sequential(

            nn.Linear(1024, 512),
            nn.ReLU(),

            nn.Linear(512, 128),
            nn.ReLU(),

            nn.Linear(128, 2)
        )

    def forward(self, x):

        return self.classifier(x)

In [90]:
import pickle

with open("data/word2idx.pkl", "rb") as f:
    word2idx = pickle.load(f)

In [91]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

encoder = SemanticEncoder().to(device)

decoder = SemanticDecoder().to(device)

question_encoder = QuestionEncoder(
    vocab_size=len(word2idx)
).to(device)

classifier = VQAClassifier().to(device)

In [92]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    list(encoder.parameters()) +
    list(decoder.parameters()) +
    list(question_encoder.parameters()) +
    list(classifier.parameters()),
    lr=1e-3
)

In [93]:
features, questions, labels = next(iter(train_loader))

features = features.to(device)
questions = questions.to(device)

labels = labels.to(device)

compressed = encoder(features)
print(compressed.shape)

complex_signal = real_to_complex(
    compressed
)
print(complex_signal.shape)

received = complex_awgn(
    complex_signal,
    snr_db=10
)

print(received.shape)

received_real = complex_to_real(
    received
)
print(received_real.shape)

reconstructed = decoder(received_real)
print(reconstructed.shape)

image_vector = reconstructed.mean(dim=1)

question_vector = question_encoder(questions)

fused = torch.cat(
    [image_vector, question_vector],
    dim=1
)

logits = classifier(
    fused
)

print(logits.shape)
print(labels.shape)

torch.Size([32, 32, 128])
torch.Size([32, 32, 64])
torch.Size([32, 32, 64])
torch.Size([32, 32, 128])
torch.Size([32, 32, 768])
torch.Size([32, 2])
torch.Size([32])


In [94]:
num_epochs = 15

for epoch in range(num_epochs):

    encoder.train()
    decoder.train()
    question_encoder.train()
    classifier.train()

    total_loss = 0
    correct = 0
    total = 0

    for features, questions, labels in train_loader:

        features = features.to(device)
        questions = questions.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        compressed = encoder(features)

        complex_signal = real_to_complex(compressed)

        received = complex_awgn(complex_signal, snr_db=10)

        received_real = complex_to_real(received)

        reconstructed = decoder(received_real)

        image_vector = reconstructed.mean(dim=1)

        question_vector = question_encoder(questions)

        fused = torch.cat(
            [image_vector, question_vector],
            dim=1
        )

        logits = classifier(
            fused
        )

        loss = criterion(
            logits,
            labels
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        predictions = torch.argmax(
            logits,
            dim=1
        )

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)

    accuracy = 100 * correct / total

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Loss: {total_loss/len(train_loader):.4f} "
        f"Accuracy: {accuracy:.2f}%"
    )

Epoch [1/15] Loss: 0.5923 Accuracy: 70.03%
Epoch [2/15] Loss: 0.4827 Accuracy: 76.95%
Epoch [3/15] Loss: 0.4196 Accuracy: 81.36%
Epoch [4/15] Loss: 0.3944 Accuracy: 84.15%
Epoch [5/15] Loss: 0.3127 Accuracy: 86.93%
Epoch [6/15] Loss: 0.2812 Accuracy: 88.62%
Epoch [7/15] Loss: 0.2366 Accuracy: 90.36%
Epoch [8/15] Loss: 0.1911 Accuracy: 92.62%
Epoch [9/15] Loss: 0.1661 Accuracy: 93.38%
Epoch [10/15] Loss: 0.1318 Accuracy: 94.54%
Epoch [11/15] Loss: 0.1087 Accuracy: 95.88%
Epoch [12/15] Loss: 0.1195 Accuracy: 95.41%
Epoch [13/15] Loss: 0.0905 Accuracy: 96.75%
Epoch [14/15] Loss: 0.0724 Accuracy: 97.39%
Epoch [15/15] Loss: 0.0653 Accuracy: 97.62%


In [95]:
val_dataset = DisasterVQADataset(
    "data/val_blip_features.pt"
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

In [96]:
encoder.eval()
decoder.eval()
question_encoder.eval()
classifier.eval()

correct = 0
total = 0

with torch.no_grad():

    for features, questions, labels in val_loader:

        features = features.to(device)
        questions = questions.to(device)
        labels = labels.to(device)

        compressed = encoder(features)

        complex_signal = real_to_complex(
            compressed
        )

        received = complex_awgn(
            complex_signal,
            snr_db=10
        )

        received_real = complex_to_real(
            received
        )

        reconstructed = decoder(
            received_real
        )

        image_vector = reconstructed.mean(
            dim=1
        )

        question_vector = question_encoder(
            questions
        )

        fused = torch.cat(
            [image_vector, question_vector],
            dim=1
        )

        logits = classifier(
            fused
        )

        predictions = torch.argmax(
            logits,
            dim=1
        )

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)

val_accuracy = 100 * correct / total

print(
    f"Validation Accuracy: {val_accuracy:.2f}%"
)

Validation Accuracy: 78.42%


In [97]:
all_predictions = []
all_labels = []

with torch.no_grad():

    for features, questions, labels in val_loader:

        features = features.to(device)
        questions = questions.to(device)
        labels = labels.to(device)

        compressed = encoder(features)

        received = awgn_channel(
            compressed,
            snr_db=10
        )

        reconstructed = decoder(received)

        image_vector = reconstructed.mean(dim=1)

        question_vector = question_encoder(
            questions
        )

        fused = torch.cat(
            [image_vector, question_vector],
            dim=1
        )

        logits = classifier(fused)

        predictions = torch.argmax(
            logits,
            dim=1
        )

        all_predictions.extend(
            predictions.cpu().tolist()
        )

        all_labels.extend(
            labels.cpu().tolist()
        )

In [98]:
train_data = torch.load("data/train_blip_features.pt")
val_data = torch.load("data/val_blip_features.pt")

train_labels = train_data["labels"]
val_labels = val_data["labels"]

print(train_labels.bincount())
print(val_labels.bincount())

tensor([ 513, 1209])
tensor([128, 303])


In [99]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(all_labels, all_predictions))

[[ 98  30]
 [ 71 232]]
